## Import libraries


In [14]:
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import os

work_dir = "/beegfs/halder/GITHUB/RESEARCH/soil-amelioration-scenarios"
data_dir = os.path.join(work_dir, "data")

## Read and clean the soil points


In [15]:
# Load shapefile once
nuts_path = Path("/beegfs/halder/DATA/DE_NUTS/DE_NUTS_3.shp")

nuts_gdf = gpd.read_file(nuts_path).to_crs(epsg=4326)

# NUTS1 regions (states)
de_nuts1_gdf = (
    nuts_gdf.loc[nuts_gdf["LEVL_CODE"] == 1, ["NUTS_NAME", "geometry"]]
    .rename(columns={"NUTS_NAME": "STATE_NAME"})
    .copy()
)

# NUTS3 regions (districts)
de_nuts3_gdf = nuts_gdf.loc[
    nuts_gdf["LEVL_CODE"] == 3, ["NUTS_ID", "NUTS_NAME", "geometry"]
].copy()

print(f"NUTS3 shape: {de_nuts3_gdf.shape}")
de_nuts3_gdf.head()

NUTS3 shape: (400, 3)


,NUTS_ID,NUTS_NAME,geometry
0,DE11B,Main-Tauber-Kreis,"POLYGON ((9.64998 49.78054, 9.6463 49.77766, 9..."
1,DE11C,Heidenheim,"MULTIPOLYGON (((10.16077 48.76496, 10.16193 48..."
2,DE11D,Ostalbkreis,"MULTIPOLYGON (((10.25676 49.05949, 10.25222 49..."
3,DE121,"Baden-Baden, Stadtkreis","MULTIPOLYGON (((8.18239 48.84184, 8.20276 48.8..."
4,DE122,"Karlsruhe, Stadtkreis","POLYGON ((8.42822 49.07141, 8.45016 49.0499, 8..."


In [19]:
# Read the soil data
soil_gdf = gpd.read_file(os.path.join(data_dir, "raw", "Site_Soil_BZE_WGS84.gpkg"))
print(soil_gdf.shape)
soil_gdf.head()

(3099, 26)


,PointID,County,Sampling_m,Sampling_y,Soil_clima,Land_use,BZE_peat,Main_soil,Specific_s,Groundwate,...,Type_of_re,Position_i,CS_0_30,CS_30_100,Longitude,Latitude,NUTS_ID,NUTS_NAME,STATE_NAME,geometry
0,2,SH,11,2015,Marsch - Nord,A,0,YE,YE/BB,0,...,V,Z,81.89,54.19,8.411608,54.859923,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.41161 54.85992)
1,3,SH,6,2016,Marsch - Nord,G,0,MD,MDn,GWS4,...,TSF,Z,71.62,59.74,8.697143,54.864382,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.69714 54.86438)
2,4,SH,8,2018,Marsch - Nord,A,0,MK,MKn,GWS4,...,TH,Z,65.60,120.4,8.765298,54.866979,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.7653 54.86698)
3,5,SH,10,2015,Marsch - Nord,G,0,RQ,p2RQ/GG-PP,GWS4,...,KSF,K,88.82,220.87,8.959032,54.863920,DEF07,Nordfriesland,Schleswig-Holstein,POINT (8.95903 54.86392)
4,6,SH,10,2015,Geest - Nord,G,0,YU,aGGe-YU,GWS4,...,TSF,Z,62.97,84.35,9.078456,54.870685,DEF07,Nordfriesland,Schleswig-Holstein,POINT (9.07846 54.87069)


## Prepare the project file


In [20]:
# Read the project file
# project_df = pd.read_csv(
#     os.path.join(data_dir, "Merged_SHP_projectdata.csv"), encoding="latin-1"
# )
# project_df.drop(
#     columns=["vlon", "vlat", "vLocationID", "vkreis", "vNUTS_NAM_1", "vState_name"],
#     inplace=True,
# )

soil_gdf_temp = soil_gdf[
    ["PointID", "Longitude", "Latitude", "NUTS_ID", "NUTS_NAME", "STATE_NAME"]
].copy()
soil_gdf_temp.rename(
    columns={
        "PointID": "vPointID",
        "Longitude": "vlon",
        "Latitude": "vlat",
        "NUTS_ID": "vNUTS_ID",
        "NUTS_NAME": "vNUTS_NAME",
        "STATE_NAME": "vSTATE_NAME",
    },
    inplace=True,
)

# project_df = pd.merge(
#     left=project_df, right=soil_gdf_temp, on=["vPointID"], how="inner"
# )
# project_df.to_csv(os.path.join(data_dir, "raw", "project_data.csv"), index=False)
# print(project_df.shape)
# project_df.head()

In [21]:
soil_gdf_temp

,vPointID,vlon,vlat,vNUTS_ID,vNUTS_NAME,vSTATE_NAME
0,2,8.411608,54.859923,DEF07,Nordfriesland,Schleswig-Holstein
1,3,8.697143,54.864382,DEF07,Nordfriesland,Schleswig-Holstein
2,4,8.765298,54.866979,DEF07,Nordfriesland,Schleswig-Holstein
3,5,8.959032,54.863920,DEF07,Nordfriesland,Schleswig-Holstein
4,6,9.078456,54.870685,DEF07,Nordfriesland,Schleswig-Holstein
...,...,...,...,...,...,...
3094,6205,7.411850,49.251206,DEB3A,"Zweibrücken, Kreisfreie Stadt",Rheinland-Pfalz
3095,6207,8.146160,49.236101,DEB3H,Südliche Weinstraße,Rheinland-Pfalz
3096,6208,8.289078,49.240987,DEB3E,Germersheim,Rheinland-Pfalz
3097,6210,8.317715,49.186116,DEB3E,Germersheim,Rheinland-Pfalz


In [ ]:
project_df = pd.read_csv(os.path.join(data_dir, "project.csv"))
print(project_df.shape)
project_df.head()

(3086, 15)


,projectid,simulationid,vColumn,vRow,vPointID,vPrecipitat,vTempMean,vRadiation,vcluster,vlon,vlat,vNUTS_ID,vNUTS_NAME,vSTATE_NAME,climate_file_exists
0,C181R22,1,181,22,2,0.00000,0.000000,0.00000,NaN,8.411608,54.859923,DEF07,Nordfriesland,Schleswig-Holstein,True
1,C200R21,1,200,21,3,15735.74583,9.611181,10972.82695,10.0,8.697143,54.864382,DEF07,Nordfriesland,Schleswig-Holstein,True
2,C204R21,1,204,21,4,15735.74583,9.611181,10972.82695,10.0,8.765298,54.866979,DEF07,Nordfriesland,Schleswig-Holstein,True
3,C217R21,1,217,21,5,15735.74583,9.611181,10972.82695,10.0,8.959032,54.863920,DEF07,Nordfriesland,Schleswig-Holstein,True
4,C224R21,1,224,21,6,15735.74583,9.611181,10972.82695,10.0,9.078456,54.870685,DEF07,Nordfriesland,Schleswig-Holstein,True


## Prepare the fertilize scenario file


In [ ]:
fertilizer_scenario = [
    {"Event": 1, "vType": "PTotal", "DVS": 0.001, "Amount": 0.214189},
    {"Event": 2, "vType": "KTotal", "DVS": 0.001, "Amount": 0.50968},
    {"Event": 3, "vType": "NTotal", "DVS": 0.25, "Amount": 6.022877},
    {"Event": 4, "vType": "PTotal", "DVS": 0.4, "Amount": 0.214189},
    {"Event": 5, "vType": "KTotal", "DVS": 0.4, "Amount": 0.50968},
    {"Event": 6, "vType": "NTotal", "DVS": 0.9, "Amount": 6.022877},
]

fertilizer_scenario_df = pd.DataFrame()

for point_id in project_df["vPointID"].unique():
    fert_scenario_df = pd.DataFrame(fertilizer_scenario)
    fert_scenario_df["location"] = point_id
    fert_scenario_df["FertilizerScenario"] = 2
    fert_scenario_df["crop"] = "winter wheat"
    fert_scenario_df = fert_scenario_df[
        ["location", "FertilizerScenario", "crop", "Event", "vType", "DVS", "Amount"]
    ]
    fertilizer_scenario_df = pd.concat(
        (fertilizer_scenario_df, fert_scenario_df), axis=0, ignore_index=True
    )

# fertilizer_scenario_df.to_csv(os.path.join(data_dir, 'fertilizer_scenario.csv'), index=False)
print(fertilizer_scenario_df.shape)
fertilizer_scenario_df.head()

(18516, 7)


,location,FertilizerScenario,crop,Event,vType,DVS,Amount
0,2,2,winter wheat,1,PTotal,0.001,0.214189
1,2,2,winter wheat,2,KTotal,0.001,0.509680
2,2,2,winter wheat,3,NTotal,0.250,6.022877
3,2,2,winter wheat,4,PTotal,0.400,0.214189
4,2,2,winter wheat,5,KTotal,0.400,0.509680


## Prepare the location file


In [ ]:
location_df = project_df[["vPointID", "vlat"]].rename(
    columns={"vPointID": "location", "vlat": "Latitude"}
)
location_df["SunInclination"] = -4
location_df["Altitude"] = 10

# location_df.to_csv(os.path.join(data_dir, 'location.csv'), index=False)
print(location_df.shape)
location_df.head()

(3086, 4)


,location,Latitude,SunInclination,Altitude
0,2,54.859923,-4,10
1,3,54.864382,-4,10
2,4,54.866979,-4,10
3,5,54.863920,-4,10
4,6,54.870685,-4,10
